# Telco Customer Churn for Marketing Retention

This notebook frames churn prediction as a marketing and retention project.

## Goal

Predict which customers are likely to churn and support a retention strategy that maximizes business value, not just model accuracy.

## What this notebook covers

- Data loading and cleaning
- Exploratory data analysis
- Baseline modeling with Logistic Regression
- Stronger modeling with XGBoost
- ROC-AUC and PR-AUC evaluation
- Threshold optimization for retention use cases
- Simple profit/ROI analysis
- Feature importance and business interpretation


## Setup

If `xgboost` is missing in this environment, uncomment the install command below and run it once.


In [1]:
# %pip install xgboost seaborn scikit-learn matplotlib pandas numpy

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


ModuleNotFoundError: No module named 'numpy'

## Step 1: Load Data

Place `Telco-Customer-Churn.csv` in `data/` under this notebooks folder, or update `DATA_PATH` below.


In [ ]:
DATA_PATH = Path("data/Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Add the dataset to the notebooks folder or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## Step 2: Basic Cleaning

We convert the target to binary, fix `TotalCharges`, and remove rows where the charge value is missing after conversion.


In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna().copy()

print(df.shape)
df.head()


## Step 3: EDA

A few interview-friendly checks:

- What is the churn rate?
- Is the dataset imbalanced?
- Which business variables look associated with churn?


In [ ]:
churn_rate = df["Churn"].value_counts(normalize=True).sort_index()
print(churn_rate)

plt.figure(figsize=(6, 4))
sns.countplot(x="Churn", data=df)
plt.title("Churn Distribution")
plt.show()

print("Average MonthlyCharges by churn:")
print(df.groupby("Churn")["MonthlyCharges"].mean())

plt.figure(figsize=(7, 4))
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges by Churn")
plt.show()

plt.figure(figsize=(8, 4))
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index")
contract_churn.plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("Churn Rate by Contract Type")
plt.ylabel("Share")
plt.legend(title="Churn")
plt.show()


### Interview talking points

- The dataset is typically imbalanced, with churn materially lower than non-churn.
- Higher `MonthlyCharges` often correlate with more churn risk.
- Month-to-month contracts tend to show meaningfully higher churn than long-term contracts.
- These patterns support targeted retention offers for high-risk, high-value customers.


## Step 4: Feature Engineering

We drop `customerID`, then keep numeric and categorical columns separate so we can use preprocessing pipelines cleanly.


In [ ]:
df = df.drop("customerID", axis=1)

X = df.drop("Churn", axis=1)
y = df["Churn"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(X_train.shape, X_test.shape)


## Step 6: Logistic Regression Baseline

A baseline is important in interviews because it shows you can compare a simple, interpretable model against a more powerful one.


In [ ]:
lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

lr_model.fit(X_train, y_train)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]


## Step 7: XGBoost Model

We fit XGBoost on an encoded feature matrix so it can handle the transformed inputs directly.


In [ ]:
X_train_xgb = pd.get_dummies(X_train, drop_first=True)
X_test_xgb = pd.get_dummies(X_test, drop_first=True)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
)

xgb_model.fit(X_train_xgb, y_train)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_xgb)[:, 1]


## Step 8: Evaluation

For imbalanced churn problems, ROC-AUC is useful, but PR-AUC is especially important because it reflects how well we rank likely churners.


In [ ]:
print("LR ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba_lr), 4))
print("XGB ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba_xgb), 4))
print("LR PR-AUC:", round(average_precision_score(y_test, y_pred_proba_lr), 4))
print("XGB PR-AUC:", round(average_precision_score(y_test, y_pred_proba_xgb), 4))


## Step 9: Threshold Optimization

Instead of using the default threshold of `0.5`, we lower it to capture more churners. This is a business decision: higher recall can be more valuable when missing a churner is expensive.


In [ ]:
threshold = 0.30
y_pred = (y_pred_proba_xgb > threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


## Step 10: Business Cost Function

Here we define a simple profit logic:

- True Positive: we correctly identify a churner and save them with a retention action
- False Negative: we miss a churner and lose that customer
- False Positive: we offer an unnecessary retention incentive


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

profit = tp * 100 + tn * 0 + fp * (-10) + fn * (-100)

print({"tn": tn, "fp": fp, "fn": fn, "tp": tp})
print("Profit:", profit)


### Optional: search for the best threshold by business value

This is often stronger than manually picking `0.30` because it shows decision optimization, not just model fitting.


In [ ]:
threshold_results = []

for t in np.arange(0.10, 0.91, 0.05):
    preds = (y_pred_proba_xgb > t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, preds).ravel()
    profit_ = tp_ * 100 + fp_ * (-10) + fn_ * (-100)
    threshold_results.append(
        {
            "threshold": round(float(t), 2),
            "tp": tp_,
            "fp": fp_,
            "fn": fn_,
            "profit": profit_,
        }
    )

threshold_df = pd.DataFrame(threshold_results).sort_values("profit", ascending=False)
threshold_df.head(10)


## Step 11: Feature Importance

This helps connect the model back to marketing strategy and customer retention actions.


In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train_xgb.columns)
top_features = importances.sort_values(ascending=False).head(10).sort_values()

plt.figure(figsize=(8, 5))
top_features.plot(kind="barh")
plt.title("Top 10 XGBoost Features")
plt.xlabel("Importance")
plt.show()

top_features


## Step 12: MLflow Tracking And Model Registry

This section shows how to track the experiment, log metrics and artifacts, and register a model version. For local use, you can keep the file-based tracking URI or point it to a dedicated MLflow server.


In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow.set_tracking_uri("file:///opt/spark/notebooks/experiments/mlruns")
mlflow.set_experiment("telco-customer-churn")

sample_input = X_test.head(5)
sample_output = lr_model.predict(sample_input)
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_name="logistic-regression-baseline"):
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("threshold", threshold)
    mlflow.log_metric("lr_roc_auc", roc_auc_score(y_test, y_pred_proba_lr))
    mlflow.log_metric("xgb_roc_auc", roc_auc_score(y_test, y_pred_proba_xgb))
    mlflow.log_metric("profit_at_threshold", profit)
    mlflow.log_dict(threshold_results, "artifacts/threshold_scan.json")

    mlflow.sklearn.log_model(
        sk_model=lr_model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input,
        registered_model_name="telco_customer_churn_lr",
    )

print("MLflow run logged. Registered model name: telco_customer_churn_lr")


## Step 13: Weights & Biases

Use this if you want hosted experiment tracking. The W&B dashboard is hosted at [https://wandb.ai](https://wandb.ai), not inside this Docker stack.

Typical flow:
1. Set `WANDB_API_KEY` from a hidden prompt using `getpass`, or export it before starting Jupyter.
2. Start a run with `wandb.init(...)`.
3. Log your churn metrics and business profit.
4. Open the project or run URL printed by the notebook.

Leave this section commented until you are ready to authenticate with your W&B account.


In [ ]:
# import wandb
#
# import getpass
# import os
#
# # Capture the API key without echoing it into notebook output.
# if not os.environ.get("WANDB_API_KEY"):
#     os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your W&B API key: ")
#
# run = wandb.init(
#     project="telco-customer-churn",
#     entity=None,  # Replace with your W&B team or username if needed.
#     job_type="training",
#     tags=["marketing", "churn", "xgboost"],
#     config={
#         "threshold": threshold,
#         "xgb_n_estimators": 200,
#         "xgb_max_depth": 4,
#     },
# )
#
# wandb.log(
#     {
#         "lr_roc_auc": roc_auc_score(y_test, y_pred_proba_lr),
#         "xgb_roc_auc": roc_auc_score(y_test, y_pred_proba_xgb),
#         "xgb_pr_auc": average_precision_score(y_test, y_pred_proba_xgb),
#         "profit": profit,
#     }
# )
#
# print("Project page:", f"https://wandb.ai/{run.entity}/{run.project}")
# print("Run page:", run.url)
#
# wandb.finish()


## Step 14: Evidently Monitoring Snapshot

A lightweight way to generate a data-quality or drift report. Here we compare the training set against the scored test set.


In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

reference_data = X_train.copy()
current_data = X_test.copy()

drift_report = Report([DataDriftPreset()])
drift_snapshot = drift_report.run(reference_data=reference_data, current_data=current_data)
drift_snapshot.save_html("reports/telco_churn_evidently_report.html")

print("Saved Evidently report to reports/telco_churn_evidently_report.html")


## Step 15: WhyLogs And WhyLabs

Use `whylogs` for local logging and optionally send profiles to WhyLabs by configuring your API credentials.


In [ ]:
import whylogs as why
from whylogs.api.writer.local import LocalWriter

scored_df = X_test.copy()
scored_df["prediction_score"] = y_pred_proba_xgb
scored_df["prediction_label"] = y_pred

profile = why.log(scored_df).profile()
writer = LocalWriter(base_dir="artifacts", base_name="telco_churn_profile.bin")
writer.write(profile)

print("Saved WhyLogs profile to artifacts/telco_churn_profile.bin")

# Optional WhyLabs upload example
# from whylabs_client import ApiClient, Configuration
# from whylogs.api.writer.whylabs import WhyLabsWriter
#
# writer = WhyLabsWriter()
# writer.write(profile.view())


## Step 16: FastAPI Scoring Contract

The local stack now includes a FastAPI starter app on `localhost:8000`. This example payload shows the kind of JSON contract you can build around the churn model.


In [ ]:
example_payload = X_test.head(1).to_dict(orient="records")
example_payload


In [ ]:
# import requests
# response = requests.post("http://localhost:8000/predict", json=example_payload)
# response.json()


## Final Story for Interviews

I built a churn prediction workflow for a marketing retention use case. I started with logistic regression as an interpretable baseline and then used XGBoost for stronger predictive power. Because churn is imbalanced, I evaluated the models with both ROC-AUC and PR-AUC. I then moved beyond the default classification threshold and optimized the decision rule using a simple business cost framework, which helped connect model output to retention ROI. Finally, I used feature importance to explain the main churn drivers, such as contract type, tenure, and monthly charges.
